# mart_user_daily_state_analysis 생성

기존 주간 마트는 보존하고, 원천 테이블을 결합해 **유저 1명 × 날짜 1일당 1행**인 2차 리텐션 분석 마트를 생성합니다.


## 데이터 마트 설계 기준

- 물리 테이블: `mart_user_daily_state_analysis` 1개
- 집계 단위: `user_id × activity_date`
- 저장 기간: 2023-07-24~2023-08-10, 총 18일
- 시간 기준: 2차 분석은 원천 저장 기준인 **UTC** 사용
- 대상 유저: 공통 기간에 Hackle 활동이 있고 숫자형 `user_id`로 확정되는 유저
- 날짜 행: 대상 유저마다 활동이 없는 날까지 18개 행 생성
- 실제 투표: `accounts_userquestionrecord`의 `user_id`, `chosen_user_id`, `created_at`만 사용
- 질문 진입: `click_question_start`; 실제 투표와 구분
- 사회적 노출: 받은 투표와 활성 동급생 수를 행동 상태와 별도 저장
- 코호트·D1·D3·D7·결과 변수: 마트에 고정 저장하지 않고 분석 SQL 또는 Python에서 계산

In [1]:
from google.cloud import bigquery

PROJECT_ID = "sns-analysis-prj"
DATASET_ID = "sns_analysis"
MART_TABLE = "mart_user_daily_state_analysis"
START_DATE = "2023-07-24"
END_DATE = "2023-08-10"
INITIAL_END_DATE = "2023-07-26"
TRANSITION_END_DATE = "2023-07-30"
OUTCOME_END_DATE = "2023-08-06"
EXPECTED_DAYS = 18
MAX_BYTES = 1100 * 1024**2

client = bigquery.Client(project=PROJECT_ID, location="asia-northeast3")


/Users/apple/DA15_Part4/sns_service_analysis/.venv312/lib/python3.12/site-packages/google/auth/_default.py:113: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)


## 1. 원천 스키마와 이벤트 키 확인

마트 생성 전에 필요한 컬럼과 핵심 이벤트 키가 실제 원천에 존재하는지 확인합니다. 질문·탐색 탭 조회는 원천에서 확인된 `view_lab_tap`을 사용합니다.


In [2]:
schema_check_sql = f"""
SELECT
    table_name,
    column_name,
    data_type
FROM `{PROJECT_ID}.{DATASET_ID}.INFORMATION_SCHEMA.COLUMNS`
WHERE table_name IN (
    'hackle_events',
    'hackle_properties',
    'accounts_user',
    'accounts_group',
    'accounts_school',
    'accounts_userquestionrecord'
)
ORDER BY table_name, ordinal_position
"""

schema_df = client.query(schema_check_sql).to_dataframe(
    create_bqstorage_client=False
)
schema_df


,table_name,column_name,data_type
0,accounts_group,id,INT64
1,accounts_group,grade,INT64
2,accounts_group,class_num,INT64
3,accounts_group,school_id,INT64
4,accounts_school,id,INT64
5,accounts_school,address,STRING
6,accounts_school,student_count,INT64
7,accounts_school,school_type,STRING
8,accounts_user,id,INT64
9,accounts_user,is_superuser,INT64


In [3]:
event_key_check_sql = f"""
SELECT
    event_key,
    COUNT(*) AS event_count
FROM `{PROJECT_ID}.{DATASET_ID}.hackle_events`
WHERE DATE(event_datetime) BETWEEN DATE('{START_DATE}') AND DATE('{END_DATE}')
  AND event_key IN (
      'launch_app',
      'click_question_start',
      'complete_question',
      'skip_question',
      'click_question_open',
      'view_lab_tap',
      'view_timeline_tap',
      'click_attendance',
      'complete_purchase'
  )
GROUP BY event_key
ORDER BY event_count DESC
"""

event_key_df = client.query(event_key_check_sql).to_dataframe(
    create_bqstorage_client=False
)
event_key_df


,event_key,event_count
0,view_timeline_tap,935557
1,view_lab_tap,820809
2,launch_app,628004
3,click_question_open,531840
4,skip_question,304448
5,click_question_start,141407
6,click_attendance,110477
7,complete_question,97656
8,complete_purchase,1541


## 2. Mart 생성 쿼리

### 생성 흐름

1. 세션별 숫자형 유저 ID를 하나로 확정합니다.
2. 공통 기간 Hackle 활동 유저를 분석 대상으로 고정하고 18일 날짜 행을 생성합니다.
3. Hackle 행동을 유저×날짜로 집계합니다.
4. 내부 투표를 투표자×수신자×날짜로 먼저 집계합니다.
5. 실제 보낸 투표와 받은 투표, 신규 상대, 상호 관계와 최근 수신 경과일을 계산합니다.
6. 회원·학급·학교 정보를 연결합니다.
7. 행동 상태와 사회적 노출을 별도 컬럼으로 저장합니다.
8. 같은 날짜·학급의 활성 유저 수를 계산합니다.

`accounts_user.friend_id_list`, 현재 `friend_count`, 포인트·pending 상태값과 `user_properties`의 학급 스냅샷은 핵심 원천으로 사용하지 않습니다.


In [4]:
mart_select_sql = f"""
-- 분석 단위: 유저 1명 × UTC 날짜 1일당 1행
-- 실제 투표: accounts_userquestionrecord만 사용
-- 받은 투표와 활성 동급생은 행동이 아닌 사회적 노출로 분리

WITH session_user AS (
    SELECT
        hp.session_id,
        ANY_VALUE(SAFE_CAST(hp.user_id AS INT64)) AS user_id
    FROM `{PROJECT_ID}.{DATASET_ID}.hackle_properties` AS hp
    WHERE hp.session_id IS NOT NULL
      AND TRIM(hp.session_id) != ''
      AND SAFE_CAST(hp.user_id AS INT64) IS NOT NULL
    GROUP BY hp.session_id
    HAVING COUNT(DISTINCT SAFE_CAST(hp.user_id AS INT64)) = 1
),

event_base AS (
    SELECT
        s.user_id,
        DATE(e.event_datetime) AS activity_date,
        e.event_datetime,
        e.event_key,
        e.session_id
    FROM `{PROJECT_ID}.{DATASET_ID}.hackle_events` AS e
    JOIN session_user AS s USING (session_id)
    WHERE DATE(e.event_datetime)
          BETWEEN DATE('{START_DATE}') AND DATE('{END_DATE}')
),

target_users AS (
    SELECT DISTINCT user_id
    FROM event_base
),

calendar AS (
    SELECT activity_date
    FROM UNNEST(
        GENERATE_DATE_ARRAY(DATE('{START_DATE}'), DATE('{END_DATE}'))
    ) AS activity_date
),

user_date_spine AS (
    SELECT
        u.user_id,
        c.activity_date
    FROM target_users AS u
    CROSS JOIN calendar AS c
),

event_daily AS (
    SELECT
        user_id,
        activity_date,
        COUNT(*) AS event_count,
        COUNT(DISTINCT session_id) AS session_count,
        COUNTIF(event_key = 'launch_app') AS launch_app_count,
        COUNTIF(event_key = 'click_question_start') AS question_start_count,
        COUNTIF(event_key = 'complete_question') AS question_complete_count,
        COUNTIF(event_key = 'skip_question') AS question_skip_count,
        COUNTIF(event_key = 'click_question_open') AS received_question_open_count,
        COUNTIF(event_key = 'view_lab_tap') AS question_tab_view_count,
        COUNTIF(event_key = 'view_timeline_tap') AS timeline_view_count,
        COUNTIF(event_key = 'click_attendance') AS attendance_click_count,
        COUNTIF(event_key = 'complete_purchase') AS purchase_count
    FROM event_base
    GROUP BY user_id, activity_date
),

user_dimension AS (
    SELECT
        u.id AS user_id,
        DATE(u.created_at) AS signup_date,
        u.gender,
        u.group_id,
        g.school_id,
        g.grade,
        g.class_num,
        sc.school_type
    FROM `{PROJECT_ID}.{DATASET_ID}.accounts_user` AS u
    LEFT JOIN `{PROJECT_ID}.{DATASET_ID}.accounts_group` AS g
        ON u.group_id = g.id
    LEFT JOIN `{PROJECT_ID}.{DATASET_ID}.accounts_school` AS sc
        ON g.school_id = sc.id
),

vote_pair_history AS (
    SELECT
        DATE(created_at) AS vote_date,
        user_id AS voter_user_id,
        chosen_user_id AS receiver_user_id,
        COUNT(*) AS pair_votes
    FROM `{PROJECT_ID}.{DATASET_ID}.accounts_userquestionrecord`
    WHERE DATE(created_at) <= DATE('{END_DATE}')
      AND user_id IS NOT NULL
      AND chosen_user_id IS NOT NULL
    GROUP BY vote_date, voter_user_id, receiver_user_id
),

pair_first_vote AS (
    SELECT
        voter_user_id,
        receiver_user_id,
        MIN(vote_date) AS first_vote_date
    FROM vote_pair_history
    GROUP BY voter_user_id, receiver_user_id
),

vote_pair_daily AS (
    SELECT
        vote_date,
        voter_user_id,
        receiver_user_id,
        pair_votes
    FROM vote_pair_history
    WHERE vote_date BETWEEN DATE('{START_DATE}') AND DATE('{END_DATE}')
),

sent_vote_daily AS (
    SELECT
        d.vote_date AS activity_date,
        d.voter_user_id AS user_id,
        SUM(d.pair_votes) AS sent_vote_count,
        COUNT(*) AS sent_unique_receiver_count,
        COUNTIF(f.first_vote_date = d.vote_date) AS new_receiver_count
    FROM vote_pair_daily AS d
    JOIN pair_first_vote AS f
        ON d.voter_user_id = f.voter_user_id
       AND d.receiver_user_id = f.receiver_user_id
    GROUP BY activity_date, user_id
),

received_vote_daily AS (
    SELECT
        d.vote_date AS activity_date,
        d.receiver_user_id AS user_id,
        SUM(d.pair_votes) AS received_vote_count,
        COUNT(*) AS received_unique_voter_count,
        COUNTIF(f.first_vote_date = d.vote_date) AS new_voter_count,
        SAFE_DIVIDE(MAX(d.pair_votes), SUM(d.pair_votes))
            AS received_vote_concentration
    FROM vote_pair_daily AS d
    JOIN pair_first_vote AS f
        ON d.voter_user_id = f.voter_user_id
       AND d.receiver_user_id = f.receiver_user_id
    GROUP BY activity_date, user_id
),

reciprocal_edges AS (
    SELECT
        p.voter_user_id AS user_id,
        p.receiver_user_id AS partner_id,
        GREATEST(p.first_vote_date, r.first_vote_date)
            AS reciprocal_start_date
    FROM pair_first_vote AS p
    JOIN pair_first_vote AS r
        ON p.voter_user_id = r.receiver_user_id
       AND p.receiver_user_id = r.voter_user_id
    WHERE p.voter_user_id != p.receiver_user_id
),

reciprocal_daily AS (
    SELECT
        s.user_id,
        s.activity_date,
        COUNT(DISTINCT r.partner_id) AS reciprocal_partner_count_to_date
    FROM user_date_spine AS s
    LEFT JOIN reciprocal_edges AS r
        ON s.user_id = r.user_id
       AND r.reciprocal_start_date <= s.activity_date
    GROUP BY s.user_id, s.activity_date
),

last_received_daily AS (
    SELECT
        s.user_id,
        s.activity_date,
        MAX(h.vote_date) AS last_received_vote_date
    FROM user_date_spine AS s
    LEFT JOIN vote_pair_history AS h
        ON s.user_id = h.receiver_user_id
       AND h.vote_date <= s.activity_date
    GROUP BY s.user_id, s.activity_date
),

daily_base AS (
    SELECT
        sp.user_id,
        sp.activity_date,
        DATE_TRUNC(sp.activity_date, WEEK(MONDAY)) AS week_start,
        CASE
            WHEN sp.activity_date <= DATE('{INITIAL_END_DATE}') THEN 'initial'
            WHEN sp.activity_date <= DATE('{TRANSITION_END_DATE}') THEN 'transition'
            WHEN sp.activity_date <= DATE('{OUTCOME_END_DATE}') THEN 'outcome'
            ELSE 'supplemental'
        END AS analysis_phase,

        u.signup_date,
        CASE
            WHEN u.signup_date IS NULL THEN NULL
            ELSE DATE_DIFF(sp.activity_date, u.signup_date, DAY)
        END AS user_tenure_days,
        u.gender,
        u.group_id,
        u.school_id,
        u.grade,
        u.class_num,
        u.school_type,
        IF(u.user_id IS NOT NULL, 1, 0) AS profile_match_flag,

        COALESCE(e.event_count, 0) AS event_count,
        COALESCE(e.session_count, 0) AS session_count,
        COALESCE(e.launch_app_count, 0) AS launch_app_count,
        COALESCE(e.question_start_count, 0) AS question_start_count,
        COALESCE(e.question_complete_count, 0) AS question_complete_count,
        COALESCE(e.question_skip_count, 0) AS question_skip_count,
        COALESCE(e.received_question_open_count, 0)
            AS received_question_open_count,
        COALESCE(e.question_tab_view_count, 0) AS question_tab_view_count,
        COALESCE(e.timeline_view_count, 0) AS timeline_view_count,
        COALESCE(e.attendance_click_count, 0) AS attendance_click_count,
        COALESCE(e.purchase_count, 0) AS purchase_count,

        COALESCE(s.sent_vote_count, 0) AS sent_vote_count,
        COALESCE(s.sent_unique_receiver_count, 0)
            AS sent_unique_receiver_count,
        COALESCE(s.new_receiver_count, 0) AS new_receiver_count,
        COALESCE(r.received_vote_count, 0) AS received_vote_count,
        COALESCE(r.received_unique_voter_count, 0)
            AS received_unique_voter_count,
        COALESCE(r.new_voter_count, 0) AS new_voter_count,
        r.received_vote_concentration,
        COALESCE(rd.reciprocal_partner_count_to_date, 0)
            AS reciprocal_partner_count_to_date,
        lr.last_received_vote_date,
        CASE
            WHEN lr.last_received_vote_date IS NULL THEN NULL
            ELSE DATE_DIFF(
                sp.activity_date,
                lr.last_received_vote_date,
                DAY
            )
        END AS days_since_last_received_vote

    FROM user_date_spine AS sp
    LEFT JOIN event_daily AS e
        ON sp.user_id = e.user_id
       AND sp.activity_date = e.activity_date
    LEFT JOIN user_dimension AS u
        ON sp.user_id = u.user_id
    LEFT JOIN sent_vote_daily AS s
        ON sp.user_id = s.user_id
       AND sp.activity_date = s.activity_date
    LEFT JOIN received_vote_daily AS r
        ON sp.user_id = r.user_id
       AND sp.activity_date = r.activity_date
    LEFT JOIN reciprocal_daily AS rd
        ON sp.user_id = rd.user_id
       AND sp.activity_date = rd.activity_date
    LEFT JOIN last_received_daily AS lr
        ON sp.user_id = lr.user_id
       AND sp.activity_date = lr.activity_date
),

daily_flags AS (
    SELECT
        b.*,
        IF(event_count > 0 OR sent_vote_count > 0, 1, 0)
            AS activity_flag,
        IF(
            question_start_count
            + question_complete_count
            + question_skip_count
            + received_question_open_count
            + question_tab_view_count
            + timeline_view_count
            + attendance_click_count
            + purchase_count
            + sent_vote_count > 0,
            1,
            0
        ) AS core_activity_flag,
        IF(question_start_count > 0, 1, 0) AS question_entry_flag,
        IF(sent_vote_count > 0, 1, 0) AS actual_vote_flag,
        IF(question_start_count > 0 AND sent_vote_count = 0, 1, 0)
            AS question_entry_no_vote_flag,
        IF(received_question_open_count > 0, 1, 0) AS reward_open_flag,
        IF(received_vote_count > 0, 1, 0) AS received_vote_flag,
        IF(attendance_click_count > 0, 1, 0) AS attendance_flag,
        IF(group_id IS NOT NULL, 1, 0) AS class_activity_available_flag
    FROM daily_base AS b
),

class_daily AS (
    SELECT
        activity_date,
        group_id,
        COUNTIF(activity_flag = 1) AS class_active_user_count
    FROM daily_flags
    WHERE group_id IS NOT NULL
    GROUP BY activity_date, group_id
),

with_anchor AS (
    SELECT
        d.*,
        MAX(
            IF(
                activity_date BETWEEN DATE('{START_DATE}')
                                  AND DATE('{INITIAL_END_DATE}'),
                activity_flag,
                0
            )
        ) OVER (PARTITION BY user_id) AS anchor_cohort_flag
    FROM daily_flags AS d
)

SELECT
    d.user_id,
    d.activity_date,
    d.week_start,
    d.analysis_phase,
    d.anchor_cohort_flag,

    d.signup_date,
    d.user_tenure_days,
    d.gender,
    d.group_id,
    d.school_id,
    d.grade,
    d.class_num,
    d.school_type,
    d.profile_match_flag,

    d.activity_flag,
    d.core_activity_flag,
    d.event_count,
    d.session_count,
    d.launch_app_count,
    d.question_start_count,
    d.question_complete_count,
    d.question_skip_count,
    d.received_question_open_count,
    d.question_tab_view_count,
    d.timeline_view_count,
    d.attendance_click_count,
    d.purchase_count,

    d.sent_vote_count,
    d.sent_unique_receiver_count,
    d.new_receiver_count,
    d.received_vote_count,
    d.received_unique_voter_count,
    d.new_voter_count,
    d.received_vote_concentration,
    d.reciprocal_partner_count_to_date,
    d.last_received_vote_date,
    d.days_since_last_received_vote,

    d.question_entry_flag,
    d.actual_vote_flag,
    d.question_entry_no_vote_flag,
    d.reward_open_flag,
    d.received_vote_flag,
    d.attendance_flag,
    d.class_activity_available_flag,

    CASE
        WHEN d.received_question_open_count > 0 THEN 'reward_opened'
        WHEN d.sent_vote_count > 0 THEN 'actual_vote'
        WHEN d.question_start_count > 0
         AND d.sent_vote_count = 0 THEN 'question_entry_no_vote'
        WHEN d.question_tab_view_count > 0
          OR d.timeline_view_count > 0 THEN 'view_centric'
        WHEN d.activity_flag = 1 THEN 'other_active'
        ELSE 'inactive'
    END AS user_behavior_state,

    c.class_active_user_count,
    CASE
        WHEN d.group_id IS NULL THEN NULL
        ELSE COALESCE(c.class_active_user_count, 0) - d.activity_flag
    END AS active_classmates_daily

FROM with_anchor AS d
LEFT JOIN class_daily AS c
    ON d.activity_date = c.activity_date
   AND d.group_id = c.group_id
"""


## 3. 예상 처리량 확인


In [5]:
dry_config = bigquery.QueryJobConfig(
    dry_run=True,
    use_query_cache=False
)
estimated = client.query(
    mart_select_sql,
    job_config=dry_config
).total_bytes_processed

print(f"예상 처리량: {estimated / 1024**2:,.2f} MiB")
if estimated > MAX_BYTES:
    raise ValueError("1.1 GiB 상한을 초과했습니다.")


예상 처리량: 731.47 MiB


## 4. Mart 생성


In [6]:
create_sql = f"""
CREATE OR REPLACE TABLE `{PROJECT_ID}.{DATASET_ID}.{MART_TABLE}`
PARTITION BY activity_date
CLUSTER BY user_id, group_id AS
{mart_select_sql}
"""

config = bigquery.QueryJobConfig(maximum_bytes_billed=MAX_BYTES)
client.query(create_sql, job_config=config).result()
print(f"{MART_TABLE} 생성 완료")


mart_user_daily_state_analysis 생성 완료


## 5. 최종 검증

아래 검증은 행 구조, 원천 합계, 상태 분리 규칙을 순서대로 확인합니다. 하나라도 `*_diff` 또는 `invalid_*`가 0이 아니면 분석을 시작하지 않습니다.


In [7]:
shape_validation_sql = f"""
WITH per_user AS (
    SELECT user_id, COUNT(*) AS row_count
    FROM `{PROJECT_ID}.{DATASET_ID}.{MART_TABLE}`
    GROUP BY user_id
),
duplicates AS (
    SELECT user_id, activity_date, COUNT(*) AS row_count
    FROM `{PROJECT_ID}.{DATASET_ID}.{MART_TABLE}`
    GROUP BY user_id, activity_date
    HAVING COUNT(*) > 1
)
SELECT
    (SELECT COUNT(*)
     FROM `{PROJECT_ID}.{DATASET_ID}.{MART_TABLE}`) AS total_rows,
    (SELECT COUNT(DISTINCT user_id)
     FROM `{PROJECT_ID}.{DATASET_ID}.{MART_TABLE}`) AS unique_users,
    (SELECT COUNT(*) FROM duplicates) AS duplicate_keys,
    MIN(row_count) AS min_rows_per_user,
    MAX(row_count) AS max_rows_per_user,
    (SELECT MIN(activity_date)
     FROM `{PROJECT_ID}.{DATASET_ID}.{MART_TABLE}`) AS min_activity_date,
    (SELECT MAX(activity_date)
     FROM `{PROJECT_ID}.{DATASET_ID}.{MART_TABLE}`) AS max_activity_date,
    (SELECT COUNT(*)
     FROM `{PROJECT_ID}.{DATASET_ID}.{MART_TABLE}`)
      - COUNT(*) * {EXPECTED_DAYS} AS expected_row_diff
FROM per_user
"""

shape_validation_df = client.query(
    shape_validation_sql,
    job_config=bigquery.QueryJobConfig(maximum_bytes_billed=MAX_BYTES)
).to_dataframe(create_bqstorage_client=False)
shape_validation_df


,total_rows,unique_users,duplicate_keys,min_rows_per_user,max_rows_per_user,min_activity_date,max_activity_date,expected_row_diff
0,3265740,181430,0,18,18,2023-07-24,2023-08-10,0


In [9]:
reconciliation_sql = f"""
WITH session_user AS (
    SELECT
        hp.session_id,
        ANY_VALUE(SAFE_CAST(hp.user_id AS INT64)) AS user_id
    FROM `{PROJECT_ID}.{DATASET_ID}.hackle_properties` AS hp
    WHERE hp.session_id IS NOT NULL
      AND TRIM(hp.session_id) != ''
      AND SAFE_CAST(hp.user_id AS INT64) IS NOT NULL
    GROUP BY hp.session_id
    HAVING COUNT(DISTINCT SAFE_CAST(hp.user_id AS INT64)) = 1
),
target_users AS (
    SELECT DISTINCT s.user_id
    FROM `{PROJECT_ID}.{DATASET_ID}.hackle_events` AS e
    JOIN session_user AS s USING (session_id)
    WHERE DATE(e.event_datetime)
          BETWEEN DATE('{START_DATE}') AND DATE('{END_DATE}')
),
source_counts AS (
    SELECT
        (
            SELECT COUNT(*)
            FROM `{PROJECT_ID}.{DATASET_ID}.hackle_events` AS e
            JOIN session_user AS s USING (session_id)
            WHERE DATE(e.event_datetime)
                  BETWEEN DATE('{START_DATE}') AND DATE('{END_DATE}')
        ) AS source_event_count,
        (
            SELECT COUNT(*)
            FROM `{PROJECT_ID}.{DATASET_ID}.accounts_userquestionrecord` AS v
            JOIN target_users AS t ON v.user_id = t.user_id
            WHERE DATE(v.created_at)
                  BETWEEN DATE('{START_DATE}') AND DATE('{END_DATE}')
        ) AS source_sent_vote_count,
        (
            SELECT COUNT(*)
            FROM `{PROJECT_ID}.{DATASET_ID}.accounts_userquestionrecord` AS v
            JOIN target_users AS t ON v.chosen_user_id = t.user_id
            WHERE DATE(v.created_at)
                  BETWEEN DATE('{START_DATE}') AND DATE('{END_DATE}')
        ) AS source_received_vote_count
),
mart_counts AS (
    SELECT
        SUM(event_count) AS mart_event_count,
        SUM(sent_vote_count) AS mart_sent_vote_count,
        SUM(received_vote_count) AS mart_received_vote_count
    FROM `{PROJECT_ID}.{DATASET_ID}.{MART_TABLE}`
)
SELECT
    s.*,
    m.*,
    m.mart_event_count - s.source_event_count AS event_count_diff,
    m.mart_sent_vote_count - s.source_sent_vote_count AS sent_vote_count_diff,
    m.mart_received_vote_count - s.source_received_vote_count
        AS received_vote_count_diff
FROM source_counts AS s
CROSS JOIN mart_counts AS m
"""

reconciliation_df = client.query(
    reconciliation_sql,
    job_config=bigquery.QueryJobConfig(maximum_bytes_billed=MAX_BYTES)
).to_dataframe(create_bqstorage_client=False)
reconciliation_df


,source_event_count,source_sent_vote_count,source_received_vote_count,mart_event_count,mart_sent_vote_count,mart_received_vote_count,event_count_diff,sent_vote_count_diff,received_vote_count_diff
0,7788571,1797,598,7788571,1797,598,0,0,0


In [10]:
state_validation_sql = f"""
SELECT
    COUNTIF(activity_flag = 0 AND user_behavior_state != 'inactive')
        AS invalid_inactive_state,
    COUNTIF(activity_flag = 1 AND user_behavior_state = 'inactive')
        AS invalid_active_state,
    COUNTIF(
        user_behavior_state = 'question_entry_no_vote'
        AND sent_vote_count > 0
    ) AS invalid_entry_no_vote_state,
    COUNTIF(received_vote_flag = 1 AND received_vote_count = 0)
        AS invalid_received_vote_flag,
    COUNTIF(received_vote_flag = 1 AND activity_flag = 0)
        AS exposure_while_inactive_rows,
    COUNTIF(user_tenure_days < 0) AS negative_tenure_rows,
    COUNTIF(profile_match_flag = 1) AS profile_matched_rows,
    COUNTIF(anchor_cohort_flag = 1) AS anchor_cohort_rows
FROM `{PROJECT_ID}.{DATASET_ID}.{MART_TABLE}`
"""

state_validation_df = client.query(
    state_validation_sql,
    job_config=bigquery.QueryJobConfig(maximum_bytes_billed=MAX_BYTES)
).to_dataframe(create_bqstorage_client=False)
state_validation_df


,invalid_inactive_state,invalid_active_state,invalid_entry_no_vote_state,invalid_received_vote_flag,exposure_while_inactive_rows,negative_tenure_rows,profile_matched_rows,anchor_cohort_rows
0,0,0,0,0,283,3611,3205404,895698


In [11]:
preview_sql = f"""
SELECT *
FROM `{PROJECT_ID}.{DATASET_ID}.{MART_TABLE}`
ORDER BY user_id, activity_date
LIMIT 100
"""

preview_df = client.query(
    preview_sql,
    job_config=bigquery.QueryJobConfig(maximum_bytes_billed=MAX_BYTES)
).to_dataframe(create_bqstorage_client=False)
preview_df


,user_id,activity_date,week_start,analysis_phase,anchor_cohort_flag,signup_date,user_tenure_days,gender,group_id,school_id,...,question_entry_flag,actual_vote_flag,question_entry_no_vote_flag,reward_open_flag,received_vote_flag,attendance_flag,class_activity_available_flag,user_behavior_state,class_active_user_count,active_classmates_daily
0,833041,2023-07-24,2023-07-24,initial,1,2023-03-31,115,F,149,314,...,0,0,0,0,0,1,1,view_centric,1,0
1,833041,2023-07-25,2023-07-24,initial,1,2023-03-31,116,F,149,314,...,0,0,0,0,0,0,1,inactive,0,0
2,833041,2023-07-26,2023-07-24,initial,1,2023-03-31,117,F,149,314,...,0,0,0,0,0,0,1,inactive,0,0
3,833041,2023-07-27,2023-07-24,transition,1,2023-03-31,118,F,149,314,...,0,0,0,0,0,1,1,view_centric,1,0
4,833041,2023-07-28,2023-07-24,transition,1,2023-03-31,119,F,149,314,...,0,0,0,0,0,0,1,inactive,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,833673,2023-07-29,2023-07-24,transition,1,2023-04-01,119,F,101,289,...,0,0,0,0,0,0,1,inactive,0,0
96,833673,2023-07-30,2023-07-24,transition,1,2023-04-01,120,F,101,289,...,0,0,0,0,0,0,1,inactive,0,0
97,833673,2023-07-31,2023-07-31,outcome,1,2023-04-01,121,F,101,289,...,0,0,0,0,0,0,1,inactive,0,0
98,833673,2023-08-01,2023-07-31,outcome,1,2023-04-01,122,F,101,289,...,0,0,0,0,0,0,1,inactive,1,1


## 6. 주요 분석 구간을 DataFrame으로 불러오기

전체 원천을 다시 조인하지 않고, 검증을 통과한 물리 마트에서 **초기 관찰 코호트의 2023-07-24~2023-08-06 행과 필요한 컬럼만** 불러옵니다. 이 셀 이후의 상태 분포·전이·세그먼트 분석은 `daily_df`로 진행합니다.


In [18]:
analysis_df_sql = f"""
SELECT
    user_id,
    activity_date,
    analysis_phase,
    signup_date,
    user_tenure_days,
    gender,
    group_id,
    school_id,
    grade,
    class_num,
    school_type,
    profile_match_flag,
    activity_flag,
    core_activity_flag,
    event_count,
    session_count,
    launch_app_count,
    question_start_count,
    question_complete_count,
    question_skip_count,
    received_question_open_count,
    question_tab_view_count,
    timeline_view_count,
    attendance_click_count,
    purchase_count,
    sent_vote_count,
    sent_unique_receiver_count,
    new_receiver_count,
    received_vote_count,
    received_unique_voter_count,
    new_voter_count,
    received_vote_concentration,
    reciprocal_partner_count_to_date,
    last_received_vote_date,
    days_since_last_received_vote,
    question_entry_flag,
    actual_vote_flag,
    question_entry_no_vote_flag,
    reward_open_flag,
    received_vote_flag,
    attendance_flag,
    class_activity_available_flag,
    user_behavior_state,
    class_active_user_count,
    active_classmates_daily
FROM `{PROJECT_ID}.{DATASET_ID}.{MART_TABLE}`
WHERE anchor_cohort_flag = 1
  AND activity_date BETWEEN DATE('{START_DATE}')
                        AND DATE('{END_DATE}')
"""

daily_df = client.query(
    analysis_df_sql,
    job_config=bigquery.QueryJobConfig(maximum_bytes_billed=MAX_BYTES)
).to_dataframe(create_bqstorage_client=False)

daily_df = daily_df.sort_values(
    ["user_id", "activity_date"]
).reset_index(drop=True)

print(f"daily_df: {daily_df.shape[0]:,}행 × {daily_df.shape[1]:,}열")
print(
    "메모리 사용량: "
    f"{daily_df.memory_usage(deep=True).sum() / 1024**2:,.1f} MiB"
)
daily_df.head()


daily_df: 895,698행 × 45열
메모리 사용량: 356.6 MiB


,user_id,activity_date,analysis_phase,signup_date,user_tenure_days,gender,group_id,school_id,grade,class_num,...,question_entry_flag,actual_vote_flag,question_entry_no_vote_flag,reward_open_flag,received_vote_flag,attendance_flag,class_activity_available_flag,user_behavior_state,class_active_user_count,active_classmates_daily
0,833041,2023-07-24,initial,2023-03-31,115,F,149,314,3,1,...,0,0,0,0,0,1,1,view_centric,1,0
1,833041,2023-07-25,initial,2023-03-31,116,F,149,314,3,1,...,0,0,0,0,0,0,1,inactive,0,0
2,833041,2023-07-26,initial,2023-03-31,117,F,149,314,3,1,...,0,0,0,0,0,0,1,inactive,0,0
3,833041,2023-07-27,transition,2023-03-31,118,F,149,314,3,1,...,0,0,0,0,0,1,1,view_centric,1,0
4,833041,2023-07-28,transition,2023-03-31,119,F,149,314,3,1,...,0,0,0,0,0,0,1,inactive,0,0


In [19]:
import pandas as pd
from datetime import date

daily_df["activity_date"] = pd.to_datetime(
    daily_df["activity_date"]
).dt.date

# 가입 전 날짜 제외
valid_day_mask = (
    daily_df["user_tenure_days"].isna()
    | (daily_df["user_tenure_days"] >= 0)
)

# 가입 후 7월 24~26일에 실제 활동한 유저만 코호트로 확정
initial_period_mask = daily_df["activity_date"].between(
    date.fromisoformat(START_DATE),
    date.fromisoformat(INITIAL_END_DATE)
)

valid_anchor_users = daily_df.loc[
    valid_day_mask
    & initial_period_mask
    & (daily_df["activity_flag"] == 1),
    "user_id"
].unique()

analysis_df = daily_df.loc[
    valid_day_mask
    & daily_df["user_id"].isin(valid_anchor_users)
].copy()

# 혼동되는 이름 정리
analysis_df["received_question_open_flag"] = (
    analysis_df["reward_open_flag"]
)

analysis_df["analysis_state"] = (
    analysis_df["user_behavior_state"]
    .replace({"reward_opened": "received_question_opened"})
)

analysis_df = analysis_df.sort_values(
    ["user_id", "activity_date"]
).reset_index(drop=True)

In [20]:
analysis_qa = pd.Series({
    "rows": len(analysis_df),
    "users": analysis_df["user_id"].nunique(),
    "min_date": analysis_df["activity_date"].min(),
    "max_date": analysis_df["activity_date"].max(),
    "duplicate_keys": analysis_df.duplicated(
        ["user_id", "activity_date"]
    ).sum(),
    "negative_tenure_rows": (
        analysis_df["user_tenure_days"] < 0
    ).sum()
})

analysis_qa

rows                        895534
users                        49757
min_date                2023-07-24
max_date                2023-08-10
duplicate_keys                   0
negative_tenure_rows             0
dtype: object

## 다음 단계

`daily_df`를 기준으로 상태 분포, 다음 날 전이, 질문 진입 후 실제 투표 여부, 투표 수신 후 확인과 재방문을 순서대로 분석합니다. 2023-08-07~08-10 보조 관찰이 필요할 때만 조회 종료일을 `END_DATE`로 확장합니다.


### 가입 전인데 실제 활동 이벤트까지 있는 행 22개 확인하기

In [12]:
tenure_check_sql = f"""
SELECT
    COUNT(*) AS negative_rows,
    COUNT(DISTINCT user_id) AS negative_users,
    COUNTIF(activity_flag = 1) AS negative_active_rows
FROM `{PROJECT_ID}.{DATASET_ID}.{MART_TABLE}`
WHERE user_tenure_days < 0
"""

tenure_check_df = client.query(
    tenure_check_sql,
    job_config=bigquery.QueryJobConfig(maximum_bytes_billed=MAX_BYTES)
).to_dataframe(create_bqstorage_client=False)

tenure_check_df

,negative_rows,negative_users,negative_active_rows
0,3611,422,22


In [13]:
negative_cause_sql = f"""
SELECT
    COUNT(*) AS negative_active_rows,
    COUNT(DISTINCT user_id) AS negative_active_users,
    COUNTIF(event_count > 0) AS hackle_event_rows,
    COUNTIF(sent_vote_count > 0) AS sent_vote_rows,
    COUNTIF(event_count > 0 AND sent_vote_count > 0) AS both_rows,
    SUM(event_count) AS total_event_count,
    SUM(sent_vote_count) AS total_sent_vote_count,
    MIN(user_tenure_days) AS min_tenure_days,
    MAX(user_tenure_days) AS max_tenure_days
FROM `{PROJECT_ID}.{DATASET_ID}.{MART_TABLE}`
WHERE user_tenure_days < 0
  AND activity_flag = 1
"""

negative_cause_df = client.query(
    negative_cause_sql,
    job_config=bigquery.QueryJobConfig(maximum_bytes_billed=MAX_BYTES)
).to_dataframe(create_bqstorage_client=False)

negative_cause_df

,negative_active_rows,negative_active_users,hackle_event_rows,sent_vote_rows,both_rows,total_event_count,total_sent_vote_count,min_tenure_days,max_tenure_days
0,22,18,22,0,0,109,0,-14,-1


In [14]:
negative_days_sql = f"""
SELECT
    user_tenure_days,
    COUNT(*) AS row_count,
    COUNT(DISTINCT user_id) AS user_count,
    COUNTIF(activity_flag = 1) AS active_rows,
    SUM(event_count) AS event_count,
    SUM(sent_vote_count) AS sent_vote_count
FROM `{PROJECT_ID}.{DATASET_ID}.{MART_TABLE}`
WHERE user_tenure_days < 0
GROUP BY user_tenure_days
ORDER BY user_tenure_days
"""

negative_days_df = client.query(
    negative_days_sql,
    job_config=bigquery.QueryJobConfig(maximum_bytes_billed=MAX_BYTES)
).to_dataframe(create_bqstorage_client=False)

negative_days_df

,user_tenure_days,row_count,user_count,active_rows,event_count,sent_vote_count
0,-17,18,18,0,0,0
1,-16,45,45,0,0,0
2,-15,71,71,0,0,0
3,-14,96,96,2,10,0
4,-13,127,127,1,3,0
5,-12,159,159,0,0,0
6,-11,183,183,0,0,0
7,-10,198,198,2,7,0
8,-9,214,214,4,18,0
9,-8,230,230,0,0,0


In [15]:
negative_detail_sql = f"""
SELECT
    user_id,
    signup_date,
    activity_date,
    user_tenure_days,
    event_count,
    session_count,
    launch_app_count,
    question_start_count,
    question_complete_count,
    received_question_open_count,
    question_tab_view_count,
    timeline_view_count,
    attendance_click_count,
    purchase_count,
    sent_vote_count,
    user_behavior_state
FROM `{PROJECT_ID}.{DATASET_ID}.{MART_TABLE}`
WHERE user_tenure_days < 0
  AND activity_flag = 1
ORDER BY user_tenure_days, user_id, activity_date
"""

negative_detail_df = client.query(
    negative_detail_sql,
    job_config=bigquery.QueryJobConfig(maximum_bytes_billed=MAX_BYTES)
).to_dataframe(create_bqstorage_client=False)

negative_detail_df

,user_id,signup_date,activity_date,user_tenure_days,event_count,session_count,launch_app_count,question_start_count,question_complete_count,received_question_open_count,question_tab_view_count,timeline_view_count,attendance_click_count,purchase_count,sent_vote_count,user_behavior_state
0,1579833,2023-08-07,2023-07-24,-14,5,1,1,0,0,0,0,0,0,0,0,other_active
1,1579889,2023-08-09,2023-07-26,-14,5,1,2,0,0,0,0,0,0,0,0,other_active
2,1579801,2023-08-06,2023-07-24,-13,3,1,1,0,0,0,0,0,0,0,0,other_active
3,1579746,2023-08-05,2023-07-26,-10,3,1,1,0,0,0,0,0,0,0,0,other_active
4,1579889,2023-08-09,2023-07-30,-10,4,1,1,0,0,0,0,0,0,0,0,other_active
5,1579745,2023-08-05,2023-07-27,-9,6,1,2,0,0,0,0,0,0,0,0,other_active
6,1579771,2023-08-06,2023-07-28,-9,4,1,1,0,0,0,0,0,0,0,0,other_active
7,1579801,2023-08-06,2023-07-28,-9,2,1,1,0,0,0,0,0,0,0,0,other_active
8,1579821,2023-08-07,2023-07-29,-9,6,1,1,0,0,0,0,0,0,0,0,other_active
9,1579855,2023-08-08,2023-08-01,-7,2,1,1,0,0,0,0,0,0,0,0,other_active


In [16]:
negative_event_sql = f"""
WITH negative_active AS (
    SELECT DISTINCT
        user_id,
        activity_date,
        signup_date
    FROM `{PROJECT_ID}.{DATASET_ID}.{MART_TABLE}`
    WHERE user_tenure_days < 0
      AND event_count > 0
),

session_user AS (
    SELECT
        hp.session_id,
        ANY_VALUE(SAFE_CAST(hp.user_id AS INT64)) AS user_id
    FROM `{PROJECT_ID}.{DATASET_ID}.hackle_properties` AS hp
    WHERE hp.session_id IS NOT NULL
      AND TRIM(hp.session_id) != ''
      AND SAFE_CAST(hp.user_id AS INT64) IS NOT NULL
    GROUP BY hp.session_id
    HAVING COUNT(DISTINCT SAFE_CAST(hp.user_id AS INT64)) = 1
)

SELECT
    n.user_id,
    n.signup_date,
    n.activity_date,
    e.event_datetime,
    e.event_key,
    e.session_id
FROM negative_active AS n
JOIN session_user AS s
    ON n.user_id = s.user_id
JOIN `{PROJECT_ID}.{DATASET_ID}.hackle_events` AS e
    ON s.session_id = e.session_id
   AND DATE(e.event_datetime) = n.activity_date
WHERE DATE(e.event_datetime)
      BETWEEN DATE('{START_DATE}') AND DATE('{END_DATE}')
ORDER BY n.user_id, e.event_datetime
"""

negative_event_df = client.query(
    negative_event_sql,
    job_config=bigquery.QueryJobConfig(maximum_bytes_billed=MAX_BYTES)
).to_dataframe(create_bqstorage_client=False)

negative_event_df

,user_id,signup_date,activity_date,event_datetime,event_key,session_id
0,1579437,2023-07-25,2023-07-24,2023-07-24 19:34:04+00:00,launch_app,33901e0e-8b1b-4242-b860-4bd16c28ef6c
1,1579437,2023-07-25,2023-07-24,2023-07-24 19:34:04+00:00,$session_start,33901e0e-8b1b-4242-b860-4bd16c28ef6c
2,1579437,2023-07-25,2023-07-24,2023-07-24 19:34:13+00:00,view_signup,33901e0e-8b1b-4242-b860-4bd16c28ef6c
3,1579437,2023-07-25,2023-07-24,2023-07-24 19:34:16+00:00,view_signup,33901e0e-8b1b-4242-b860-4bd16c28ef6c
4,1579437,2023-07-25,2023-07-24,2023-07-24 19:34:35+00:00,view_signup,33901e0e-8b1b-4242-b860-4bd16c28ef6c
...,...,...,...,...,...,...
104,1579889,2023-08-09,2023-08-08,2023-08-08 08:33:44+00:00,view_signup,F20F1886-69E0-4F8A-8613-594290682553
105,1579889,2023-08-09,2023-08-08,2023-08-08 08:33:47+00:00,view_signup,F20F1886-69E0-4F8A-8613-594290682553
106,1579889,2023-08-09,2023-08-08,2023-08-08 08:33:52+00:00,view_signup,F20F1886-69E0-4F8A-8613-594290682553
107,1579889,2023-08-09,2023-08-08,2023-08-08 08:33:53+00:00,view_signup,F20F1886-69E0-4F8A-8613-594290682553


- 분석 대상 중 가입 전 날짜 행이 존재하는 유저는 422명이었으며, 이 중 18명이 가입 전에 앱을 실행하거나 가입 화면을 조회했다. 해당 활동은 22개 유저-일, 총 109개 Hackle 이벤트로 확인됐다.
- 마트 집계에는 문제가 없으며, 가입 전 앱 실행·가입 화면 조회가 최종 유저 ID에 연결된 사례였다. 회원가입 후 리텐션 분석에서는 가입 전 날짜를 제외하고 초기 코호트를 다시 확정한다.